# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

The dataset contains protein abundance, modification, and sequence data from mass spectrometry analysis of isolated human mast cell extracellular vesicles.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets, their fields, and column IDs. All items are referenced by their `@id` as per best practice for Croissant datasets.

In [ ]:
# List all record sets and their fields

if not hasattr(metadata, 'record_sets'):
    # fallback for newest mlcroissant versions
    record_sets = dataset.record_sets
else:
    record_sets = metadata.record_sets

if not record_sets:
    # Sometimes Croissant v1 schema lists in 'recordSet' for backwards compat
    record_sets = getattr(metadata, 'recordSet', [])

if not record_sets or len(record_sets) == 0:
    print("No record sets found in this dataset's Croissant metadata.")
else:
    for rs in record_sets:
        print(f'RecordSet: {rs["@id"]}')
        fields = rs.get('fields', []) or rs.get('field', [])
        for fld in fields:
            print(f'  Field: {fld["@id"]} -- {fld.get("name","(no name)")}, type: {fld.get("dataType","(no type)")}')
        # List columns (if applicable)
        columns = rs.get('columns', []) or rs.get('column', [])
        for col in columns:
            print(f'  Column: {col["@id"]} -- {col.get("name","(no name)")}, type: {col.get("dataType","(no type)")}')
        print()

# If no record sets, print an informative message
# Users can refer to Croissant in the browser as a fallback to inspect the schema

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use record set and field `@id`s as discovered above.

> **Note:** For the FAIR² dataset, common record set candidates may include protein abundance tables, modifications or other experimental sheets.

In [ ]:
# Discover available record set @ids
# If your dataset only has one record set, you may hardcode or inspect IDs from previous overview cell.

record_sets_metadata = getattr(metadata, 'record_sets', None)
if record_sets_metadata is None or record_sets_metadata == []:
    record_sets_metadata = getattr(metadata, 'recordSet', [])

record_set_ids = []
for rs in record_sets_metadata:
    if isinstance(rs, dict) and '@id' in rs:
        record_set_ids.append(rs['@id'])
    elif isinstance(rs, str):
        record_set_ids.append(rs)

print('Record set @ids:', record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

for rs_id, df in dataframes.items():
    print(f'RecordSet: {rs_id} -- Columns:', df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group using selected numeric and categorical fields. 

**Important:** All field/column access is by their `@id` in accordance with the Croissant schema and best practice.

Below, we assume there is an abundance or count field (e.g. a peptide count). Modify `numeric_field_id` and `group_field_id` as needed.

In [ ]:
# Select record set and field ids (by @id) from previous outputs

# If you know field/column IDs, define them below. Otherwise, revisit the overview output.
if len(dataframes) == 0:
    print("No dataframes loaded. Please ensure record sets are available in metadata.")
else:
    # Example record set and fields; replace with real @id for your dataset:
    target_record_set_id = record_set_ids[0]  # Update if needed
    df = dataframes[target_record_set_id]

    print(f"Available columns in {target_record_set_id}:\n", df.columns.tolist())

    # Try to auto-select numeric and group fields
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    if not numeric_candidates:
        # Try to coerce common numeric columns
        for col in df.columns:
            try:
                _ = pd.to_numeric(df[col])
                numeric_candidates.append(col)
            except:
                continue
    numeric_field_id = numeric_candidates[0] if numeric_candidates else df.columns[0]  # fallback

    group_field_candidates = [col for col in df.columns if col != numeric_field_id]
    group_field_id = group_field_candidates[0] if group_field_candidates else numeric_field_id

    print(f"Using numeric field: {numeric_field_id}")
    print(f"Using group field: {group_field_id}")

    # EDA: Filtering
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df[numeric_field_id] > df[numeric_field_id].mean()]
    print(f"Filtered records where {numeric_field_id} > mean:")
    display(filtered_df.head())

    # Normalization (Z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        )
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Plot distributions or relationships in the data. Adjust fields as desired for your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field_id in filtered_df.columns:
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True, color='cornflowerblue')
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.show()

    # Barplot of means by group
    if 'grouped_df' in locals() and group_field_id in grouped_df.columns:
        plt.figure(figsize=(12,4))
        # Only plot if reasonable number of groups
        if len(grouped_df) < 40:
            sns.barplot(
                x=group_field_id,
                y=numeric_field_id,
                data=grouped_df,
                palette='mako'
            )
            plt.xticks(rotation=75)
            plt.title(f"Mean {numeric_field_id} by {group_field_id}")
            plt.tight_layout()
            plt.show()
        else:
            print(f'Group count {len(grouped_df)} too large for labeled barplot. Showing only summary.')
            display(grouped_df.head())

## 6. Conclusion
In this notebook, we've loaded, inspected, transformed, and visualized the FAIR² mass spectrometry EV dataset using the `mlcroissant` library.

_You can expand on this template by incorporating more advanced analytics, machine learning, or dataset curation steps as needed. All data manipulation and access should continue to reference Croissant entities by their `@id` for robust, schema-driven workflows._